# Experiment: Discrete Map GIS


## 0. 开头与公共准备

### 0.1 Notebook 目标与结构说明

本 notebook 在 `exp_map_0321.ipynb` 的 Rulkov map 数据基础上，改用数据驱动的 `GIS` 主流程来比较四种状态 `CS / GS / Q / D` 的宏微观效率。

本实验的核心设定有三点：

1. 微观观测不再只取恒等映射，而是采用“恒等项 + 平方项”的观测函数。
2. 先在观测层拟合微观 `GIS`，再由 `SVD` 路线构造粗粒化矩阵 `W` 得到宏观变量。
3. 结合预测误差、`Gamma`、`J_\alpha`、`D`、`N` 与 `CE` 判断宏观表示是否相对微观表示更有效。

整体主线可以写成

$$
\mathbf{x}_t
\rightarrow
\mathbf{o}_t = g(\mathbf{x}_t)
\rightarrow
(\mathbf{A}_o, \boldsymbol{\Sigma}_o)
\rightarrow
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t
\rightarrow
(\mathbf{A}_z, \boldsymbol{\Sigma}_z)
\rightarrow
\mathrm{CE} = J_{\alpha,z} - J_{\alpha,o}.
$$

下面按照“解释块 + 代码块”交替的方式，对四种状态分别完成同一套分析流程，最后再做统一比较。


In [1]:
import sys
from pathlib import Path

REPO_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / 'tools' / 'tools.py').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing tools/tools.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import types

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError:
    plotly_module = types.ModuleType('plotly')
    express_module = types.ModuleType('plotly.express')
    graph_objects_module = types.ModuleType('plotly.graph_objects')
    plotly_module.express = express_module
    plotly_module.graph_objects = graph_objects_module
    sys.modules['plotly'] = plotly_module
    sys.modules['plotly.express'] = express_module
    sys.modules['plotly.graph_objects'] = graph_objects_module

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from tools import (
    generate_two_population_neuron_data,
    plot_neuron_analysis_combo,
    extract_state_matrix_from_rulkov_data,
    add_gaussian_noise,
    prepare_time_pairs,
    fit_linear_gis_from_pairs,
    compute_gis_metrics,
    compute_prediction_errors,
    compute_ce_from_micro_macro,
    select_macro_rank,
    build_w_from_svd,
    apply_coarse_graining,
    summarize_pipeline_results,
)


### 0.2 公共依赖、绘图风格与统一参数

本实验延续参考 notebook 的数据来源与状态参数，但为了控制维度规模并突出 Rulkov map 的快变量，本 notebook 默认只取 `x` 通道作为底层观测，再在其上加入平方项。

因此微观观测函数写为

$$
g(\mathbf{x}_t)=\bigl[\mathbf{x}_t,\ \mathbf{x}_t^{\odot 2}\bigr],
$$

其中 $\odot 2$ 表示逐分量平方。这样既保留了原始神经元群体的线性信息，也显式加入了最基本的非线性项。

统一参数包括时间尺度 $\tau$、噪声强度、`ridge` 与 `eps` 等；这些参数对四种状态保持一致，从而保证横向比较主要反映状态差异，而不是建模设置差异。


In [2]:
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei',
    'SimHei',
    'Noto Sans CJK SC',
    'Source Han Sans SC',
    'Arial Unicode MS',
    'DejaVu Sans',
]

HEATMAP_CMAP = 'vlag'

config = {
    'experiment_name': 'exp_map_gis',
    'include_x': True,
    'include_y': False,
    'use_transient': True,
    'tau': 1,
    'burn_in': 0,
    'stride': 1,
    'alpha': 1.0,
    'eps': 1e-10,
    'ridge': 1e-10,
    'noise_scale': 0.03,
    'noise_seed': 42,
    'horizons': (1, 3, 5),
    'manual_ranks': {'CS': None, 'GS': None, 'Q': None, 'D': None},
    'selected_raw_indices': [0, 25, 100, 150],
    'selected_obs_indices': [0, 25, 100, 150, 200, 225],
    'max_ticks': 18,
    'curve_window': 160,
    'states': {
        'CS': {'label': 'Complete Synchronization (CS)', 'n_a': 100, 'n_b': 100, 'alpha_a': 4.6, 'alpha_b': 4.6, 'sigma_a': 0.225, 'sigma_b': 0.225, 'mu': 0.001, 'gamma': 0.005, 'epsilon': 0.002, 'T': 5000, 'transients': 3000, 'seed': 101, 'x0_a': -1.0, 'x0_b': -1.0, 'y0_a': -3.5, 'y0_b': -3.5},
        'GS': {'label': 'Generalized Synchronization (GS)', 'n_a': 100, 'n_b': 100, 'alpha_a': 4.6, 'alpha_b': 4.6, 'sigma_a': 0.225, 'sigma_b': 0.225, 'mu': 0.001, 'gamma': 0.06, 'epsilon': 0.02, 'T': 5000, 'transients': 1000, 'seed': 100, 'x0_a': -1.0, 'x0_b': -1.2, 'y0_a': -3.5, 'y0_b': -3.7},
        'Q': {'label': 'Chimera State (Q)', 'n_a': 100, 'n_b': 100, 'alpha_a': 4.6, 'alpha_b': 4.6, 'sigma_a': 0.225, 'sigma_b': 0.225, 'mu': 0.001, 'gamma': 0.005, 'epsilon': 0.002, 'T': 5000, 'transients': 3000, 'seed': 103, 'x0_a': -1.0, 'x0_b': None, 'y0_a': -3.5, 'y0_b': None},
        'D': {'label': 'Desynchronization (D)', 'n_a': 100, 'n_b': 100, 'alpha_a': 4.6, 'alpha_b': 4.6, 'sigma_a': 0.225, 'sigma_b': 0.225, 'mu': 0.001, 'gamma': 0.005, 'epsilon': 0.002, 'T': 5000, 'transients': 3000, 'seed': 155, 'x0_a': None, 'x0_b': None, 'y0_a': None, 'y0_b': None},
    },
}

state_order = ['CS', 'GS', 'Q', 'D']
state_runs = {}
summary_rows = []

config


{'experiment_name': 'exp_map_gis',
 'include_x': True,
 'include_y': False,
 'use_transient': True,
 'tau': 1,
 'burn_in': 0,
 'stride': 1,
 'alpha': 1.0,
 'eps': 1e-10,
 'ridge': 1e-10,
 'noise_scale': 0.03,
 'noise_seed': 42,
 'horizons': (1, 3, 5),
 'manual_ranks': {'CS': None, 'GS': None, 'Q': None, 'D': None},
 'selected_raw_indices': [0, 25, 100, 150],
 'selected_obs_indices': [0, 25, 100, 150, 200, 225],
 'max_ticks': 18,
 'curve_window': 160,
 'states': {'CS': {'label': 'Complete Synchronization (CS)',
   'n_a': 100,
   'n_b': 100,
   'alpha_a': 4.6,
   'alpha_b': 4.6,
   'sigma_a': 0.225,
   'sigma_b': 0.225,
   'mu': 0.001,
   'gamma': 0.005,
   'epsilon': 0.002,
   'T': 5000,
   'transients': 3000,
   'seed': 101,
   'x0_a': -1.0,
   'x0_b': -1.0,
   'y0_a': -3.5,
   'y0_b': -3.5},
  'GS': {'label': 'Generalized Synchronization (GS)',
   'n_a': 100,
   'n_b': 100,
   'alpha_a': 4.6,
   'alpha_b': 4.6,
   'sigma_a': 0.225,
   'sigma_b': 0.225,
   'mu': 0.001,
   'gamma': 0.06

### 0.3 公共函数区（本实验特有的，不包括复用 `tools.py` 里面的函数）

这一块只放本 notebook 自己特有的辅助函数，主要完成五件事：

1. 构造“恒等项 + 平方项”的观测函数。
2. 让高维矩阵热力图的标签自动稀疏显示。
3. 把微观与宏观误差、指标整理成更易读的表格。
4. 让宏微观曲线对比时只挑选少量代表性微观通道。
5. 在每个状态结束时生成统一格式的结果摘要。


In [3]:
def sparse_labels(labels, max_ticks=18):
    if labels is None:
        return False
    labels = list(labels)
    if len(labels) <= max_ticks:
        return labels
    step = int(np.ceil(len(labels) / max_ticks))
    return [label if idx % step == 0 else '' for idx, label in enumerate(labels)]


def plot_matrix_heatmap(matrix, title, row_labels=None, col_labels=None, center=0.0, figsize=(6.2, 5.2), cmap=HEATMAP_CMAP, max_ticks=18):
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(matrix, ax=ax, cmap=cmap, center=center, square=matrix.shape[0] == matrix.shape[1], xticklabels=sparse_labels(col_labels, max_ticks=max_ticks), yticklabels=sparse_labels(row_labels, max_ticks=max_ticks), cbar_kws={'shrink': 0.75})
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90, labelsize=6.5)
    ax.tick_params(axis='y', rotation=0, labelsize=6.5)
    plt.tight_layout()
    plt.show()


def standardize_for_plot(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)


def pick_valid_indices(candidates, upper_bound, fallback_count=4):
    valid = [idx for idx in candidates if 0 <= idx < upper_bound]
    if valid:
        return valid
    if upper_bound <= fallback_count:
        return list(range(upper_bound))
    return list(np.linspace(0, upper_bound - 1, fallback_count, dtype=int))


def build_identity_square_observables(x_data, feature_names):
    x_data = np.asarray(x_data, dtype=float)
    obs = np.concatenate([x_data, x_data ** 2], axis=1)
    raw_names = list(feature_names)
    sq_names = [f'{name}^2' for name in raw_names]
    obs_names = raw_names + sq_names
    obs_centered = obs - np.mean(obs, axis=0, keepdims=True)
    return {'data': np.asarray(obs_centered, dtype=float), 'feature_names': obs_names, 'micro_dim': obs_centered.shape[1], 'base_dim': x_data.shape[1]}


def choose_macro_rank(values, manual_r=None, eps=1e-10, relative_cutoff=0.7, max_rank=12):
    values = np.real_if_close(np.asarray(values, dtype=float).ravel())
    positive = values[values > eps]
    if len(positive) == 0:
        raise ValueError('没有有效奇异值，无法选择宏观维度')
    positive = np.sort(positive)[::-1]
    if manual_r is not None:
        r = int(max(1, min(manual_r, len(positive))))
        return r, {'mode': 'manual', 'manual_r': manual_r, 'relative_cutoff': relative_cutoff, 'max_rank': max_rank, 'selected_r': r, 'effective_rank': int(len(positive)), 'used_values': positive}
    relative = positive / positive[0]
    candidate_count = int(np.sum(relative >= relative_cutoff))
    candidate_count = max(1, candidate_count)
    r = min(candidate_count, max_rank, len(positive))
    return r, {'mode': 'relative_threshold', 'manual_r': manual_r, 'relative_cutoff': relative_cutoff, 'max_rank': max_rank, 'selected_r': int(r), 'effective_rank': int(len(positive)), 'used_values': positive}


def build_metric_table(metrics):
    return pd.DataFrame({'metric': ['Gamma', 'log_Gamma', 'J_alpha', 'D', 'N'], 'value': [metrics['Gamma'], metrics['log_Gamma'], metrics['J_alpha'], metrics['D'], metrics['N']]})


def build_error_table(errors):
    return pd.DataFrame({'horizon': list(errors.keys()), 'mean_error': [errors[h]['mean_error'] for h in errors.keys()]})

def plot_dual_singular_spectra(forward_values, backward_values, title, forward_label='前向谱', backward_label='后向谱'):
    forward_values = np.asarray(forward_values, dtype=float)
    backward_values = np.asarray(backward_values, dtype=float)
    n = int(min(len(forward_values), len(backward_values)))
    x = np.arange(1, n + 1)
    width = 0.38

    fig, ax = plt.subplots(figsize=(8.4, 4.4))
    ax.bar(x - width / 2, forward_values[:n], width=width, color='tab:blue', alpha=0.45, label=forward_label)
    ax.bar(x + width / 2, backward_values[:n], width=width, color='tab:orange', alpha=0.45, label=backward_label)
    ax.plot(x - width / 2, forward_values[:n], color='tab:blue', marker='o', linewidth=1.8)
    ax.plot(x + width / 2, backward_values[:n], color='tab:orange', marker='o', linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel('奇异值序号')
    ax.set_ylabel('奇异值')
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_prediction_comparison(predictions, targets, names, title, steps=120):
    n_show = min(len(names), predictions.shape[1], 4)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    window = min(steps, len(predictions))
    for idx in range(n_show):
        ax.plot(targets[:window, idx], linewidth=1.3, label=f'true: {names[idx]}')
        ax.plot(predictions[:window, idx], '--', linewidth=1.8, label=f'pred: {names[idx]}')
    ax.set_title(title)
    ax.set_xlabel('样本编号')
    ax.set_ylabel('取值')
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.show()


def plot_macro_micro_curves(micro_data, macro_data, micro_names, macro_names, title, max_micro=4, max_macro=4, steps=160):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    window = min(steps, len(micro_data), len(macro_data))
    for idx in range(min(max_micro, micro_data.shape[1])):
        ax.plot(standardize_for_plot(micro_data[:window, idx]), linewidth=1.0, alpha=0.8, label=f'微观: {micro_names[idx]}')
    for idx in range(min(max_macro, macro_data.shape[1])):
        ax.plot(standardize_for_plot(macro_data[:window, idx]), '--', linewidth=2.0, label=f'宏观: {macro_names[idx]}')
    ax.set_title(title)
    ax.set_xlabel('时间步')
    ax.set_ylabel('标准化取值')
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.show()


def summarize_state_outputs(state_key, state_cfg, sync_metrics, micro_metrics, macro_metrics, micro_errors, macro_errors, ce_result, rank_meta):
    return {
        'state': state_key,
        'label': state_cfg['label'],
        'sync_state': sync_metrics.get('sync_state'),
        'R_a': sync_metrics.get('R_a'),
        'R_b': sync_metrics.get('R_b'),
        'R_t': sync_metrics.get('R_t'),
        'R_delta': sync_metrics.get('R_delta'),
        'micro_dim': micro_metrics['dimension'],
        'macro_dim': macro_metrics['dimension'],
        'selected_r': rank_meta['selected_r'],
        'micro_J_alpha': micro_metrics['J_alpha'],
        'macro_J_alpha': macro_metrics['J_alpha'],
        'micro_log_Gamma': micro_metrics['log_Gamma'],
        'macro_log_Gamma': macro_metrics['log_Gamma'],
        'micro_E1': micro_errors[1]['mean_error'],
        'macro_E1': macro_errors[1]['mean_error'],
        'CE': ce_result['CE'],
    }


## 第一部分：CS数据


### 1.1 数据生成加上噪音，并绘图

这一块延续 `exp_map_0321.ipynb` 的状态参数，先生成对应的 Rulkov map 原始轨迹，再在提取出的 `x` 通道微观矩阵上加入高斯噪声。

这样做对应研究框架里的“数据准备”步骤：我们先得到统一的原始时间序列 $\{\mathbf{x}_t\}$，再把噪声视作有限观测与未建模扰动的经验代理。这里保留参考 notebook 的同步指标，并额外画出少量代表性通道的“干净/含噪”曲线对比。


In [4]:
state_key = 'CS'
state_cfg = config['states'][state_key]
state_args = dict(state_cfg)
state_args.pop('label', None)

data = generate_two_population_neuron_data(**state_args)
raw_pack = extract_state_matrix_from_rulkov_data(data, include_x=config['include_x'], include_y=config['include_y'], use_transient=config['use_transient'])

x_clean = np.asarray(raw_pack['x_data_raw'], dtype=float)
state_names = list(raw_pack['state_names'])
time_data = np.asarray(raw_pack['time_data'])
sync_metrics = raw_pack['sync_metrics']
noise_seed = config['noise_seed'] + state_cfg['seed']
x_noisy = add_gaussian_noise(x_clean, noise_scale=config['noise_scale'], random_state=noise_seed)['noisy_data']
raw_indices = pick_valid_indices(config['selected_raw_indices'], x_clean.shape[1])

state_runs[state_key] = {'state_cfg': state_cfg, 'data': data, 'raw_pack': raw_pack, 'x_clean': x_clean, 'x_noisy': x_noisy, 'state_names': state_names, 'time_data': time_data, 'sync_metrics': sync_metrics, 'raw_indices': raw_indices}

print('状态:', state_key, '-', state_cfg['label'])
print('原始矩阵形状:', x_clean.shape)
print('噪声种子:', noise_seed)
print('同步指标:', sync_metrics)
display(pd.DataFrame([{'state': state_key, 'sync_state': sync_metrics.get('sync_state'), 'R_a': sync_metrics.get('R_a'), 'R_b': sync_metrics.get('R_b'), 'R_t': sync_metrics.get('R_t'), 'R_delta': sync_metrics.get('R_delta')}]))

plot_neuron_analysis_combo(data, figsize=(18, 7))

fig, ax = plt.subplots(figsize=(10, 4.2))
window = min(config['curve_window'], x_clean.shape[0])
for idx in raw_indices[:4]:
    ax.plot(x_clean[:window, idx], linewidth=1.1, label=f'clean: {state_names[idx]}')
    ax.plot(x_noisy[:window, idx], '--', linewidth=1.1, alpha=0.9, label=f'noisy: {state_names[idx]}')
ax.set_title(f'{state_key}: 原始通道加噪前后对比')
ax.set_xlabel('时间步')
ax.set_ylabel('取值')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


TypeError: generate_two_population_neuron_data() got an unexpected keyword argument 'label'

### 1.2 观测函数、时间尺度与样本配对

这里采用本实验指定的观测函数

$$
\mathbf{o}_t = g(\mathbf{x}_t)=\bigl[\mathbf{x}_t,\ \mathbf{x}_t^{\odot 2}\bigr],
$$

并在固定时间尺度 $\tau=1$ 下构造样本对 $(\mathbf{o}_t, \mathbf{o}_{t+\tau})$。平方项的作用是把最基本的非线性信息直接显式写入观测层，使后续线性 `GIS` 能在更丰富的特征空间中拟合有效动力学。


In [ ]:
run = state_runs[state_key]
obs_pack = build_identity_square_observables(run['x_noisy'], run['state_names'])
obs_data = obs_pack['data']
feature_names = obs_pack['feature_names']
X_now, X_next = prepare_time_pairs(obs_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
obs_indices = pick_valid_indices(config['selected_obs_indices'], obs_data.shape[1], fallback_count=6)

run.update({'obs_pack': obs_pack, 'obs_data': obs_data, 'feature_names': feature_names, 'X_now': X_now, 'X_next': X_next, 'obs_indices': obs_indices})

print('观测层维度:', obs_pack['micro_dim'])
print('样本对形状:', X_now.shape, X_next.shape)
print('前 10 个观测标签:', feature_names[:10])
display(pd.DataFrame([{'state': state_key, 'base_dim': obs_pack['base_dim'], 'micro_dim': obs_pack['micro_dim'], 'tau': config['tau'], 'pair_count': X_now.shape[0]}]))


### 1.3 微观层 A/K_raw 与 Sigma 的拟合

根据数据驱动 `GIS` 的主流程，在观测层拟合

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}_o \mathbf{o}_t + \boldsymbol{\varepsilon}_t^{(o)},
\qquad
\boldsymbol{\Sigma}_o = \mathrm{Cov}(\mathbf{e}_t^{(o)}).
$$

这里同时保留 `A` 与 `K_raw`。在当前最小二乘实现下，两者本质上是同一经验回归关系的两种记法；热力图主要帮助观察微观层中哪些通道之间存在更强的耦合结构。


In [ ]:
run = state_runs[state_key]
micro_fit = fit_linear_gis_from_pairs(run['X_now'], run['X_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_micro = micro_fit['A']
K_raw_micro = micro_fit['K_raw']
Sigma_micro = micro_fit['Sigma']
run.update({'micro_fit': micro_fit, 'A_micro': A_micro, 'K_raw_micro': K_raw_micro, 'Sigma_micro': Sigma_micro})

print('A_micro shape:', A_micro.shape)
print('K_raw_micro shape:', K_raw_micro.shape)
print('Sigma_micro shape:', Sigma_micro.shape)
plot_matrix_heatmap(A_micro, f'{state_key}: 微观层 A', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(K_raw_micro, f'{state_key}: 微观层 K_raw', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_micro, f'{state_key}: 微观层 Sigma', row_labels=run['feature_names'], col_labels=run['feature_names'], center=None, max_ticks=config['max_ticks'])


### 1.4 微观层预测与误差

得到微观层矩阵 $\mathbf{A}_o$ 后，可以做单步和多步滚动预测：

$$
\hat{\mathbf{o}}_{t+k\tau \mid t} = \mathbf{A}_o^k \mathbf{o}_t.
$$

这里重点看两件事：第一，观测层模型是否具有足够好的闭合性；第二，不同状态在相同观测函数下的误差规模是否存在明显差异。


In [ ]:
run = state_runs[state_key]
micro_errors = compute_prediction_errors(run['A_micro'], run['obs_data'], tau=config['tau'], horizons=config['horizons'])
run['micro_errors'] = micro_errors

display(build_error_table(micro_errors))
selected_names = [run['feature_names'][idx] for idx in run['obs_indices'][:4]]
selected_preds = micro_errors[1]['predictions'][:, run['obs_indices'][:4]]
selected_targets = micro_errors[1]['targets'][:, run['obs_indices'][:4]]
plot_prediction_comparison(selected_preds, selected_targets, selected_names, f'{state_key}: 微观层单步预测对比')


### 1.5 微观层 GIS 指标

微观层 `GIS` 指标由 $(\mathbf{A}_o, \boldsymbol{\Sigma}_o)$ 决定，其中核心量包括

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}_o^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}_o^\top \boldsymbol{\Sigma}_o^{-1} \mathbf{A}_o),
$$

以及

$$
J_\alpha = \frac{1}{m} \log \Gamma_\alpha^{\mathrm{GIS}}.
$$

其中 $D$ 描述前向确定性，$N$ 描述后向非简并性，$J_\alpha$ 则是维度归一化后的综合效率。


In [ ]:
run = state_runs[state_key]
micro_metrics = compute_gis_metrics(run['A_micro'], run['Sigma_micro'], alpha=config['alpha'], eps=config['eps'])
run['micro_metrics'] = micro_metrics

display(build_metric_table(micro_metrics))
plot_dual_singular_spectra(
    micro_metrics['sv_forward'],
    micro_metrics['sv_backward'],
    title=f'{state_key}: 微观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 1.6 宏观维度选择与 W 的构造

宏观层的关键在于先选维数 $r$，再构造粗粒化矩阵 $\mathbf{W}$。这里沿用 `SVD` 路线，根据微观层后向谱自动选择候选维数，并构造

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t,
\qquad
\mathbf{W} \in \mathbb{R}^{r \times m}.
$$

`W` 的每一行都可以理解为一个宏观变量在微观观测基上的线性组合。由于微观维数较高，这里的标签会自动稀疏显示。


In [ ]:
run = state_runs[state_key]
manual_r = config['manual_ranks'][state_key]
r, rank_meta = choose_macro_rank(run['micro_metrics']['sv_backward'], manual_r=manual_r, eps=config['eps'])
w_result = build_w_from_svd(run['A_micro'], run['Sigma_micro'], r=r, alpha=config['alpha'], eps=config['eps'], mode='two_stage')
W = w_result['W']
macro_names = [f'z_{i+1}' for i in range(r)]
run.update({'rank_meta': rank_meta, 'w_result': w_result, 'W': W, 'macro_names': macro_names})

print('宏观维度选择结果:', rank_meta)
plot_matrix_heatmap(np.abs(W), f'{state_key}: |W| 热力图', row_labels=macro_names, col_labels=run['feature_names'], center=None, figsize=(7.5, 3.8), cmap='Blues', max_ticks=config['max_ticks'])


### 1.7 宏观数据生成

有了粗粒化矩阵之后，就可以把微观观测投影到宏观层，得到

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t.
$$

这一块只负责生成宏观数据和宏观层样本对，并画出少量宏观变量的时间曲线，帮助直观理解粗粒化后的主导模态。


In [ ]:
run = state_runs[state_key]
z_data = apply_coarse_graining(run['W'], run['obs_data'])
Z_now, Z_next = prepare_time_pairs(z_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
run.update({'z_data': z_data, 'Z_now': Z_now, 'Z_next': Z_next})

print('宏观数据形状:', z_data.shape)
print('宏观样本对形状:', Z_now.shape, Z_next.shape)

fig, ax = plt.subplots(figsize=(10, 4.0))
for idx in range(min(z_data.shape[1], 4)):
    ax.plot(z_data[:config['curve_window'], idx], linewidth=1.6, label=run['macro_names'][idx])
ax.set_title(f'{state_key}: 宏观时间序列')
ax.set_xlabel('时间步')
ax.set_ylabel('宏观变量')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 1.8 宏观层拟合

在宏观层上重复一次微观层建模步骤，拟合

$$
\mathbf{z}_{t+\tau} \approx \mathbf{A}_z \mathbf{z}_t + \boldsymbol{\varepsilon}_t^{(z)}.
$$

如果粗粒化是合理的，那么宏观层不仅维度更低，而且应当仍能保留较好的动力学闭合性。


In [ ]:
run = state_runs[state_key]
macro_fit = fit_linear_gis_from_pairs(run['Z_now'], run['Z_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_macro = macro_fit['A']
Sigma_macro = macro_fit['Sigma']
run.update({'macro_fit': macro_fit, 'A_macro': A_macro, 'Sigma_macro': Sigma_macro})

plot_matrix_heatmap(A_macro, f'{state_key}: 宏观层 A', row_labels=run['macro_names'], col_labels=run['macro_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_macro, f'{state_key}: 宏观层 Sigma', row_labels=run['macro_names'], col_labels=run['macro_names'], center=None, max_ticks=config['max_ticks'])


### 1.9 宏观层预测、误差

宏观层也做与微观层完全对应的单步与多步预测。这里需要重点比较两个问题：

1. 宏观误差是否显著大于微观误差。
2. 降维之后是否仍保持较好的滚动预测稳定性。

如果 `CE` 虽然提高，但预测误差明显恶化，那么该宏观表示仍然不能算是一个可信的因果涌现候选。


In [ ]:
run = state_runs[state_key]
macro_errors = compute_prediction_errors(run['A_macro'], run['z_data'], tau=config['tau'], horizons=config['horizons'])
run['macro_errors'] = macro_errors

display(build_error_table(macro_errors))
plot_prediction_comparison(macro_errors[1]['predictions'], macro_errors[1]['targets'], run['macro_names'], f'{state_key}: 宏观层单步预测对比')


### 1.10 宏观层GIS指标

宏观层同样需要计算 `Gamma`、`log Gamma`、`J_\alpha`、`D`、`N`。只有在两层都得到完整指标之后，`CE` 才能被解释为真正的“宏观效率增益”，而不是单纯的降维结果。


In [ ]:
run = state_runs[state_key]
macro_metrics = compute_gis_metrics(run['A_macro'], run['Sigma_macro'], alpha=config['alpha'], eps=config['eps'])
run['macro_metrics'] = macro_metrics

display(build_metric_table(macro_metrics))
plot_dual_singular_spectra(
    macro_metrics['sv_forward'],
    macro_metrics['sv_backward'],
    title=f'{state_key}: 宏观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 1.11 CE值+宏微观曲线对比

因果涌现强度这里直接定义为

$$
\mathrm{CE} = J_{\alpha,z} - J_{\alpha,o}.
$$

若 `CE > 0`，说明宏观层在“单位维度效率”意义上优于微观层；但是否值得接受，还要结合前面预测误差是否显著恶化一起看。

这一块把 `CE` 数值和宏微观代表性曲线放到一起展示，便于同时观察“效率提升”和“动态形态是否保留”。


In [ ]:
run = state_runs[state_key]
ce_result = compute_ce_from_micro_macro(run['micro_metrics'], run['macro_metrics'])
run['ce_result'] = ce_result

comparison_df_local = pd.DataFrame([
    {'layer': 'micro', 'J_alpha': run['micro_metrics']['J_alpha'], 'log_Gamma': run['micro_metrics']['log_Gamma'], 'E1': run['micro_errors'][1]['mean_error']},
    {'layer': 'macro', 'J_alpha': run['macro_metrics']['J_alpha'], 'log_Gamma': run['macro_metrics']['log_Gamma'], 'E1': run['macro_errors'][1]['mean_error']},
    {'layer': 'macro - micro', 'J_alpha': ce_result['delta_J'], 'log_Gamma': ce_result['delta_log_Gamma'], 'E1': run['macro_errors'][1]['mean_error'] - run['micro_errors'][1]['mean_error']},
])
print(f'{state_key} 的 CE = {ce_result["CE"]:.6f}')
display(comparison_df_local)

micro_curve_indices = run['obs_indices'][:min(4, len(run['obs_indices']))]
micro_curve_data = run['obs_data'][:, micro_curve_indices]
micro_curve_names = [run['feature_names'][idx] for idx in micro_curve_indices]
plot_macro_micro_curves(micro_curve_data, run['z_data'], micro_curve_names, run['macro_names'], f'{state_key}: 宏微观曲线对比', max_micro=4, max_macro=4, steps=config['curve_window'])


### 1.12 结果汇总

最后把本状态的参数、同步指标、宏微观误差和 `GIS` 指标整理为统一摘要。这样一方面便于本部分回顾，另一方面也能直接服务于最后的四状态横向比较。


In [ ]:
run = state_runs[state_key]
summary_dict, summary_row = summarize_pipeline_results(
    config={'experiment_name': config['experiment_name'], 'state': state_key, 'tau': config['tau'], 'alpha': config['alpha'], 'noise_scale': config['noise_scale'], 'observable': 'identity + square (x-channel)'},
    micro_fit=run['micro_fit'],
    macro_fit=run['macro_fit'],
    micro_metrics=run['micro_metrics'],
    macro_metrics=run['macro_metrics'],
    prediction_results={'micro_errors': run['micro_errors'], 'macro_errors': run['macro_errors']},
    ce_result=run['ce_result'],
    extra={'W': run['W'], 'rank_meta': run['rank_meta'], 'sync_metrics': run['sync_metrics']},
)
state_row = summarize_state_outputs(state_key, run['state_cfg'], run['sync_metrics'], run['micro_metrics'], run['macro_metrics'], run['micro_errors'], run['macro_errors'], run['ce_result'], run['rank_meta'])
run['summary_dict'] = summary_dict
run['summary_row'] = state_row
summary_rows.append(state_row)

display(pd.DataFrame([state_row]))


## 第二部分：GS数据


### 2.1 数据生成加上噪音，并绘图

这一块延续 `exp_map_0321.ipynb` 的状态参数，先生成对应的 Rulkov map 原始轨迹，再在提取出的 `x` 通道微观矩阵上加入高斯噪声。

这样做对应研究框架里的“数据准备”步骤：我们先得到统一的原始时间序列 $\{\mathbf{x}_t\}$，再把噪声视作有限观测与未建模扰动的经验代理。这里保留参考 notebook 的同步指标，并额外画出少量代表性通道的“干净/含噪”曲线对比。


In [ ]:
state_key = 'GS'
state_cfg = config['states'][state_key]
state_args = dict(state_cfg)
state_args.pop('label', None)

data = generate_two_population_neuron_data(**state_args)
raw_pack = extract_state_matrix_from_rulkov_data(data, include_x=config['include_x'], include_y=config['include_y'], use_transient=config['use_transient'])

x_clean = np.asarray(raw_pack['x_data_raw'], dtype=float)
state_names = list(raw_pack['state_names'])
time_data = np.asarray(raw_pack['time_data'])
sync_metrics = raw_pack['sync_metrics']
noise_seed = config['noise_seed'] + state_cfg['seed']
x_noisy = add_gaussian_noise(x_clean, noise_scale=config['noise_scale'], random_state=noise_seed)['noisy_data']
raw_indices = pick_valid_indices(config['selected_raw_indices'], x_clean.shape[1])

state_runs[state_key] = {'state_cfg': state_cfg, 'data': data, 'raw_pack': raw_pack, 'x_clean': x_clean, 'x_noisy': x_noisy, 'state_names': state_names, 'time_data': time_data, 'sync_metrics': sync_metrics, 'raw_indices': raw_indices}

print('状态:', state_key, '-', state_cfg['label'])
print('原始矩阵形状:', x_clean.shape)
print('噪声种子:', noise_seed)
print('同步指标:', sync_metrics)
display(pd.DataFrame([{'state': state_key, 'sync_state': sync_metrics.get('sync_state'), 'R_a': sync_metrics.get('R_a'), 'R_b': sync_metrics.get('R_b'), 'R_t': sync_metrics.get('R_t'), 'R_delta': sync_metrics.get('R_delta')}]))

plot_neuron_analysis_combo(data, figsize=(18, 7))

fig, ax = plt.subplots(figsize=(10, 4.2))
window = min(config['curve_window'], x_clean.shape[0])
for idx in raw_indices[:4]:
    ax.plot(x_clean[:window, idx], linewidth=1.1, label=f'clean: {state_names[idx]}')
    ax.plot(x_noisy[:window, idx], '--', linewidth=1.1, alpha=0.9, label=f'noisy: {state_names[idx]}')
ax.set_title(f'{state_key}: 原始通道加噪前后对比')
ax.set_xlabel('时间步')
ax.set_ylabel('取值')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 2.2 观测函数、时间尺度与样本配对

这里采用本实验指定的观测函数

$$
\mathbf{o}_t = g(\mathbf{x}_t)=\bigl[\mathbf{x}_t,\ \mathbf{x}_t^{\odot 2}\bigr],
$$

并在固定时间尺度 $\tau=1$ 下构造样本对 $(\mathbf{o}_t, \mathbf{o}_{t+\tau})$。平方项的作用是把最基本的非线性信息直接显式写入观测层，使后续线性 `GIS` 能在更丰富的特征空间中拟合有效动力学。


In [ ]:
run = state_runs[state_key]
obs_pack = build_identity_square_observables(run['x_noisy'], run['state_names'])
obs_data = obs_pack['data']
feature_names = obs_pack['feature_names']
X_now, X_next = prepare_time_pairs(obs_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
obs_indices = pick_valid_indices(config['selected_obs_indices'], obs_data.shape[1], fallback_count=6)

run.update({'obs_pack': obs_pack, 'obs_data': obs_data, 'feature_names': feature_names, 'X_now': X_now, 'X_next': X_next, 'obs_indices': obs_indices})

print('观测层维度:', obs_pack['micro_dim'])
print('样本对形状:', X_now.shape, X_next.shape)
print('前 10 个观测标签:', feature_names[:10])
display(pd.DataFrame([{'state': state_key, 'base_dim': obs_pack['base_dim'], 'micro_dim': obs_pack['micro_dim'], 'tau': config['tau'], 'pair_count': X_now.shape[0]}]))


### 2.3 微观层 A/K_raw 与 Sigma 的拟合

根据数据驱动 `GIS` 的主流程，在观测层拟合

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}_o \mathbf{o}_t + \boldsymbol{\varepsilon}_t^{(o)},
\qquad
\boldsymbol{\Sigma}_o = \mathrm{Cov}(\mathbf{e}_t^{(o)}).
$$

这里同时保留 `A` 与 `K_raw`。在当前最小二乘实现下，两者本质上是同一经验回归关系的两种记法；热力图主要帮助观察微观层中哪些通道之间存在更强的耦合结构。


In [ ]:
run = state_runs[state_key]
micro_fit = fit_linear_gis_from_pairs(run['X_now'], run['X_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_micro = micro_fit['A']
K_raw_micro = micro_fit['K_raw']
Sigma_micro = micro_fit['Sigma']
run.update({'micro_fit': micro_fit, 'A_micro': A_micro, 'K_raw_micro': K_raw_micro, 'Sigma_micro': Sigma_micro})

print('A_micro shape:', A_micro.shape)
print('K_raw_micro shape:', K_raw_micro.shape)
print('Sigma_micro shape:', Sigma_micro.shape)
plot_matrix_heatmap(A_micro, f'{state_key}: 微观层 A', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(K_raw_micro, f'{state_key}: 微观层 K_raw', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_micro, f'{state_key}: 微观层 Sigma', row_labels=run['feature_names'], col_labels=run['feature_names'], center=None, max_ticks=config['max_ticks'])


### 2.4 微观层预测与误差

得到微观层矩阵 $\mathbf{A}_o$ 后，可以做单步和多步滚动预测：

$$
\hat{\mathbf{o}}_{t+k\tau \mid t} = \mathbf{A}_o^k \mathbf{o}_t.
$$

这里重点看两件事：第一，观测层模型是否具有足够好的闭合性；第二，不同状态在相同观测函数下的误差规模是否存在明显差异。


In [ ]:
run = state_runs[state_key]
micro_errors = compute_prediction_errors(run['A_micro'], run['obs_data'], tau=config['tau'], horizons=config['horizons'])
run['micro_errors'] = micro_errors

display(build_error_table(micro_errors))
selected_names = [run['feature_names'][idx] for idx in run['obs_indices'][:4]]
selected_preds = micro_errors[1]['predictions'][:, run['obs_indices'][:4]]
selected_targets = micro_errors[1]['targets'][:, run['obs_indices'][:4]]
plot_prediction_comparison(selected_preds, selected_targets, selected_names, f'{state_key}: 微观层单步预测对比')


### 2.5 微观层 GIS 指标

微观层 `GIS` 指标由 $(\mathbf{A}_o, \boldsymbol{\Sigma}_o)$ 决定，其中核心量包括

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}_o^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}_o^\top \boldsymbol{\Sigma}_o^{-1} \mathbf{A}_o),
$$

以及

$$
J_\alpha = \frac{1}{m} \log \Gamma_\alpha^{\mathrm{GIS}}.
$$

其中 $D$ 描述前向确定性，$N$ 描述后向非简并性，$J_\alpha$ 则是维度归一化后的综合效率。


In [ ]:
run = state_runs[state_key]
micro_metrics = compute_gis_metrics(run['A_micro'], run['Sigma_micro'], alpha=config['alpha'], eps=config['eps'])
run['micro_metrics'] = micro_metrics

display(build_metric_table(micro_metrics))
plot_dual_singular_spectra(
    micro_metrics['sv_forward'],
    micro_metrics['sv_backward'],
    title=f'{state_key}: 微观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 2.6 宏观维度选择与 W 的构造

宏观层的关键在于先选维数 $r$，再构造粗粒化矩阵 $\mathbf{W}$。这里沿用 `SVD` 路线，根据微观层后向谱自动选择候选维数，并构造

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t,
\qquad
\mathbf{W} \in \mathbb{R}^{r \times m}.
$$

`W` 的每一行都可以理解为一个宏观变量在微观观测基上的线性组合。由于微观维数较高，这里的标签会自动稀疏显示。


In [ ]:
run = state_runs[state_key]
manual_r = config['manual_ranks'][state_key]
r, rank_meta = choose_macro_rank(run['micro_metrics']['sv_backward'], manual_r=manual_r, eps=config['eps'])
w_result = build_w_from_svd(run['A_micro'], run['Sigma_micro'], r=r, alpha=config['alpha'], eps=config['eps'], mode='two_stage')
W = w_result['W']
macro_names = [f'z_{i+1}' for i in range(r)]
run.update({'rank_meta': rank_meta, 'w_result': w_result, 'W': W, 'macro_names': macro_names})

print('宏观维度选择结果:', rank_meta)
plot_matrix_heatmap(np.abs(W), f'{state_key}: |W| 热力图', row_labels=macro_names, col_labels=run['feature_names'], center=None, figsize=(7.5, 3.8), cmap='Blues', max_ticks=config['max_ticks'])


### 2.7 宏观数据生成

有了粗粒化矩阵之后，就可以把微观观测投影到宏观层，得到

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t.
$$

这一块只负责生成宏观数据和宏观层样本对，并画出少量宏观变量的时间曲线，帮助直观理解粗粒化后的主导模态。


In [ ]:
run = state_runs[state_key]
z_data = apply_coarse_graining(run['W'], run['obs_data'])
Z_now, Z_next = prepare_time_pairs(z_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
run.update({'z_data': z_data, 'Z_now': Z_now, 'Z_next': Z_next})

print('宏观数据形状:', z_data.shape)
print('宏观样本对形状:', Z_now.shape, Z_next.shape)

fig, ax = plt.subplots(figsize=(10, 4.0))
for idx in range(min(z_data.shape[1], 4)):
    ax.plot(z_data[:config['curve_window'], idx], linewidth=1.6, label=run['macro_names'][idx])
ax.set_title(f'{state_key}: 宏观时间序列')
ax.set_xlabel('时间步')
ax.set_ylabel('宏观变量')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 2.8 宏观层拟合

在宏观层上重复一次微观层建模步骤，拟合

$$
\mathbf{z}_{t+\tau} \approx \mathbf{A}_z \mathbf{z}_t + \boldsymbol{\varepsilon}_t^{(z)}.
$$

如果粗粒化是合理的，那么宏观层不仅维度更低，而且应当仍能保留较好的动力学闭合性。


In [ ]:
run = state_runs[state_key]
macro_fit = fit_linear_gis_from_pairs(run['Z_now'], run['Z_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_macro = macro_fit['A']
Sigma_macro = macro_fit['Sigma']
run.update({'macro_fit': macro_fit, 'A_macro': A_macro, 'Sigma_macro': Sigma_macro})

plot_matrix_heatmap(A_macro, f'{state_key}: 宏观层 A', row_labels=run['macro_names'], col_labels=run['macro_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_macro, f'{state_key}: 宏观层 Sigma', row_labels=run['macro_names'], col_labels=run['macro_names'], center=None, max_ticks=config['max_ticks'])


### 2.9 宏观层预测、误差

宏观层也做与微观层完全对应的单步与多步预测。这里需要重点比较两个问题：

1. 宏观误差是否显著大于微观误差。
2. 降维之后是否仍保持较好的滚动预测稳定性。

如果 `CE` 虽然提高，但预测误差明显恶化，那么该宏观表示仍然不能算是一个可信的因果涌现候选。


In [ ]:
run = state_runs[state_key]
macro_errors = compute_prediction_errors(run['A_macro'], run['z_data'], tau=config['tau'], horizons=config['horizons'])
run['macro_errors'] = macro_errors

display(build_error_table(macro_errors))
plot_prediction_comparison(macro_errors[1]['predictions'], macro_errors[1]['targets'], run['macro_names'], f'{state_key}: 宏观层单步预测对比')


### 2.10 宏观层GIS指标

宏观层同样需要计算 `Gamma`、`log Gamma`、`J_\alpha`、`D`、`N`。只有在两层都得到完整指标之后，`CE` 才能被解释为真正的“宏观效率增益”，而不是单纯的降维结果。


In [ ]:
run = state_runs[state_key]
macro_metrics = compute_gis_metrics(run['A_macro'], run['Sigma_macro'], alpha=config['alpha'], eps=config['eps'])
run['macro_metrics'] = macro_metrics

display(build_metric_table(macro_metrics))
plot_dual_singular_spectra(
    macro_metrics['sv_forward'],
    macro_metrics['sv_backward'],
    title=f'{state_key}: 宏观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 2.11 CE值+宏微观曲线对比

因果涌现强度这里直接定义为

$$
\mathrm{CE} = J_{\alpha,z} - J_{\alpha,o}.
$$

若 `CE > 0`，说明宏观层在“单位维度效率”意义上优于微观层；但是否值得接受，还要结合前面预测误差是否显著恶化一起看。

这一块把 `CE` 数值和宏微观代表性曲线放到一起展示，便于同时观察“效率提升”和“动态形态是否保留”。


In [ ]:
run = state_runs[state_key]
ce_result = compute_ce_from_micro_macro(run['micro_metrics'], run['macro_metrics'])
run['ce_result'] = ce_result

comparison_df_local = pd.DataFrame([
    {'layer': 'micro', 'J_alpha': run['micro_metrics']['J_alpha'], 'log_Gamma': run['micro_metrics']['log_Gamma'], 'E1': run['micro_errors'][1]['mean_error']},
    {'layer': 'macro', 'J_alpha': run['macro_metrics']['J_alpha'], 'log_Gamma': run['macro_metrics']['log_Gamma'], 'E1': run['macro_errors'][1]['mean_error']},
    {'layer': 'macro - micro', 'J_alpha': ce_result['delta_J'], 'log_Gamma': ce_result['delta_log_Gamma'], 'E1': run['macro_errors'][1]['mean_error'] - run['micro_errors'][1]['mean_error']},
])
print(f'{state_key} 的 CE = {ce_result["CE"]:.6f}')
display(comparison_df_local)

micro_curve_indices = run['obs_indices'][:min(4, len(run['obs_indices']))]
micro_curve_data = run['obs_data'][:, micro_curve_indices]
micro_curve_names = [run['feature_names'][idx] for idx in micro_curve_indices]
plot_macro_micro_curves(micro_curve_data, run['z_data'], micro_curve_names, run['macro_names'], f'{state_key}: 宏微观曲线对比', max_micro=4, max_macro=4, steps=config['curve_window'])


### 2.12 结果汇总

最后把本状态的参数、同步指标、宏微观误差和 `GIS` 指标整理为统一摘要。这样一方面便于本部分回顾，另一方面也能直接服务于最后的四状态横向比较。


In [ ]:
run = state_runs[state_key]
summary_dict, summary_row = summarize_pipeline_results(
    config={'experiment_name': config['experiment_name'], 'state': state_key, 'tau': config['tau'], 'alpha': config['alpha'], 'noise_scale': config['noise_scale'], 'observable': 'identity + square (x-channel)'},
    micro_fit=run['micro_fit'],
    macro_fit=run['macro_fit'],
    micro_metrics=run['micro_metrics'],
    macro_metrics=run['macro_metrics'],
    prediction_results={'micro_errors': run['micro_errors'], 'macro_errors': run['macro_errors']},
    ce_result=run['ce_result'],
    extra={'W': run['W'], 'rank_meta': run['rank_meta'], 'sync_metrics': run['sync_metrics']},
)
state_row = summarize_state_outputs(state_key, run['state_cfg'], run['sync_metrics'], run['micro_metrics'], run['macro_metrics'], run['micro_errors'], run['macro_errors'], run['ce_result'], run['rank_meta'])
run['summary_dict'] = summary_dict
run['summary_row'] = state_row
summary_rows.append(state_row)

display(pd.DataFrame([state_row]))


## 第三部分：Q部分


### 3.1 数据生成加上噪音，并绘图

这一块延续 `exp_map_0321.ipynb` 的状态参数，先生成对应的 Rulkov map 原始轨迹，再在提取出的 `x` 通道微观矩阵上加入高斯噪声。

这样做对应研究框架里的“数据准备”步骤：我们先得到统一的原始时间序列 $\{\mathbf{x}_t\}$，再把噪声视作有限观测与未建模扰动的经验代理。这里保留参考 notebook 的同步指标，并额外画出少量代表性通道的“干净/含噪”曲线对比。


In [ ]:
state_key = 'Q'
state_cfg = config['states'][state_key]
state_args = dict(state_cfg)
state_args.pop('label', None)

data = generate_two_population_neuron_data(**state_args)
raw_pack = extract_state_matrix_from_rulkov_data(data, include_x=config['include_x'], include_y=config['include_y'], use_transient=config['use_transient'])

x_clean = np.asarray(raw_pack['x_data_raw'], dtype=float)
state_names = list(raw_pack['state_names'])
time_data = np.asarray(raw_pack['time_data'])
sync_metrics = raw_pack['sync_metrics']
noise_seed = config['noise_seed'] + state_cfg['seed']
x_noisy = add_gaussian_noise(x_clean, noise_scale=config['noise_scale'], random_state=noise_seed)['noisy_data']
raw_indices = pick_valid_indices(config['selected_raw_indices'], x_clean.shape[1])

state_runs[state_key] = {'state_cfg': state_cfg, 'data': data, 'raw_pack': raw_pack, 'x_clean': x_clean, 'x_noisy': x_noisy, 'state_names': state_names, 'time_data': time_data, 'sync_metrics': sync_metrics, 'raw_indices': raw_indices}

print('状态:', state_key, '-', state_cfg['label'])
print('原始矩阵形状:', x_clean.shape)
print('噪声种子:', noise_seed)
print('同步指标:', sync_metrics)
display(pd.DataFrame([{'state': state_key, 'sync_state': sync_metrics.get('sync_state'), 'R_a': sync_metrics.get('R_a'), 'R_b': sync_metrics.get('R_b'), 'R_t': sync_metrics.get('R_t'), 'R_delta': sync_metrics.get('R_delta')}]))

plot_neuron_analysis_combo(data, figsize=(18, 7))

fig, ax = plt.subplots(figsize=(10, 4.2))
window = min(config['curve_window'], x_clean.shape[0])
for idx in raw_indices[:4]:
    ax.plot(x_clean[:window, idx], linewidth=1.1, label=f'clean: {state_names[idx]}')
    ax.plot(x_noisy[:window, idx], '--', linewidth=1.1, alpha=0.9, label=f'noisy: {state_names[idx]}')
ax.set_title(f'{state_key}: 原始通道加噪前后对比')
ax.set_xlabel('时间步')
ax.set_ylabel('取值')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 3.2 观测函数、时间尺度与样本配对

这里采用本实验指定的观测函数

$$
\mathbf{o}_t = g(\mathbf{x}_t)=\bigl[\mathbf{x}_t,\ \mathbf{x}_t^{\odot 2}\bigr],
$$

并在固定时间尺度 $\tau=1$ 下构造样本对 $(\mathbf{o}_t, \mathbf{o}_{t+\tau})$。平方项的作用是把最基本的非线性信息直接显式写入观测层，使后续线性 `GIS` 能在更丰富的特征空间中拟合有效动力学。


In [ ]:
run = state_runs[state_key]
obs_pack = build_identity_square_observables(run['x_noisy'], run['state_names'])
obs_data = obs_pack['data']
feature_names = obs_pack['feature_names']
X_now, X_next = prepare_time_pairs(obs_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
obs_indices = pick_valid_indices(config['selected_obs_indices'], obs_data.shape[1], fallback_count=6)

run.update({'obs_pack': obs_pack, 'obs_data': obs_data, 'feature_names': feature_names, 'X_now': X_now, 'X_next': X_next, 'obs_indices': obs_indices})

print('观测层维度:', obs_pack['micro_dim'])
print('样本对形状:', X_now.shape, X_next.shape)
print('前 10 个观测标签:', feature_names[:10])
display(pd.DataFrame([{'state': state_key, 'base_dim': obs_pack['base_dim'], 'micro_dim': obs_pack['micro_dim'], 'tau': config['tau'], 'pair_count': X_now.shape[0]}]))


### 3.3 微观层 A/K_raw 与 Sigma 的拟合

根据数据驱动 `GIS` 的主流程，在观测层拟合

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}_o \mathbf{o}_t + \boldsymbol{\varepsilon}_t^{(o)},
\qquad
\boldsymbol{\Sigma}_o = \mathrm{Cov}(\mathbf{e}_t^{(o)}).
$$

这里同时保留 `A` 与 `K_raw`。在当前最小二乘实现下，两者本质上是同一经验回归关系的两种记法；热力图主要帮助观察微观层中哪些通道之间存在更强的耦合结构。


In [ ]:
run = state_runs[state_key]
micro_fit = fit_linear_gis_from_pairs(run['X_now'], run['X_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_micro = micro_fit['A']
K_raw_micro = micro_fit['K_raw']
Sigma_micro = micro_fit['Sigma']
run.update({'micro_fit': micro_fit, 'A_micro': A_micro, 'K_raw_micro': K_raw_micro, 'Sigma_micro': Sigma_micro})

print('A_micro shape:', A_micro.shape)
print('K_raw_micro shape:', K_raw_micro.shape)
print('Sigma_micro shape:', Sigma_micro.shape)
plot_matrix_heatmap(A_micro, f'{state_key}: 微观层 A', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(K_raw_micro, f'{state_key}: 微观层 K_raw', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_micro, f'{state_key}: 微观层 Sigma', row_labels=run['feature_names'], col_labels=run['feature_names'], center=None, max_ticks=config['max_ticks'])


### 3.4 微观层预测与误差

得到微观层矩阵 $\mathbf{A}_o$ 后，可以做单步和多步滚动预测：

$$
\hat{\mathbf{o}}_{t+k\tau \mid t} = \mathbf{A}_o^k \mathbf{o}_t.
$$

这里重点看两件事：第一，观测层模型是否具有足够好的闭合性；第二，不同状态在相同观测函数下的误差规模是否存在明显差异。


In [ ]:
run = state_runs[state_key]
micro_errors = compute_prediction_errors(run['A_micro'], run['obs_data'], tau=config['tau'], horizons=config['horizons'])
run['micro_errors'] = micro_errors

display(build_error_table(micro_errors))
selected_names = [run['feature_names'][idx] for idx in run['obs_indices'][:4]]
selected_preds = micro_errors[1]['predictions'][:, run['obs_indices'][:4]]
selected_targets = micro_errors[1]['targets'][:, run['obs_indices'][:4]]
plot_prediction_comparison(selected_preds, selected_targets, selected_names, f'{state_key}: 微观层单步预测对比')


### 3.5 微观层 GIS 指标

微观层 `GIS` 指标由 $(\mathbf{A}_o, \boldsymbol{\Sigma}_o)$ 决定，其中核心量包括

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}_o^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}_o^\top \boldsymbol{\Sigma}_o^{-1} \mathbf{A}_o),
$$

以及

$$
J_\alpha = \frac{1}{m} \log \Gamma_\alpha^{\mathrm{GIS}}.
$$

其中 $D$ 描述前向确定性，$N$ 描述后向非简并性，$J_\alpha$ 则是维度归一化后的综合效率。


In [ ]:
run = state_runs[state_key]
micro_metrics = compute_gis_metrics(run['A_micro'], run['Sigma_micro'], alpha=config['alpha'], eps=config['eps'])
run['micro_metrics'] = micro_metrics

display(build_metric_table(micro_metrics))
plot_dual_singular_spectra(
    micro_metrics['sv_forward'],
    micro_metrics['sv_backward'],
    title=f'{state_key}: 微观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 3.6 宏观维度选择与 W 的构造

宏观层的关键在于先选维数 $r$，再构造粗粒化矩阵 $\mathbf{W}$。这里沿用 `SVD` 路线，根据微观层后向谱自动选择候选维数，并构造

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t,
\qquad
\mathbf{W} \in \mathbb{R}^{r \times m}.
$$

`W` 的每一行都可以理解为一个宏观变量在微观观测基上的线性组合。由于微观维数较高，这里的标签会自动稀疏显示。


In [ ]:
run = state_runs[state_key]
manual_r = config['manual_ranks'][state_key]
r, rank_meta = choose_macro_rank(run['micro_metrics']['sv_backward'], manual_r=manual_r, eps=config['eps'])
w_result = build_w_from_svd(run['A_micro'], run['Sigma_micro'], r=r, alpha=config['alpha'], eps=config['eps'], mode='two_stage')
W = w_result['W']
macro_names = [f'z_{i+1}' for i in range(r)]
run.update({'rank_meta': rank_meta, 'w_result': w_result, 'W': W, 'macro_names': macro_names})

print('宏观维度选择结果:', rank_meta)
plot_matrix_heatmap(np.abs(W), f'{state_key}: |W| 热力图', row_labels=macro_names, col_labels=run['feature_names'], center=None, figsize=(7.5, 3.8), cmap='Blues', max_ticks=config['max_ticks'])


### 3.7 宏观数据生成

有了粗粒化矩阵之后，就可以把微观观测投影到宏观层，得到

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t.
$$

这一块只负责生成宏观数据和宏观层样本对，并画出少量宏观变量的时间曲线，帮助直观理解粗粒化后的主导模态。


In [ ]:
run = state_runs[state_key]
z_data = apply_coarse_graining(run['W'], run['obs_data'])
Z_now, Z_next = prepare_time_pairs(z_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
run.update({'z_data': z_data, 'Z_now': Z_now, 'Z_next': Z_next})

print('宏观数据形状:', z_data.shape)
print('宏观样本对形状:', Z_now.shape, Z_next.shape)

fig, ax = plt.subplots(figsize=(10, 4.0))
for idx in range(min(z_data.shape[1], 4)):
    ax.plot(z_data[:config['curve_window'], idx], linewidth=1.6, label=run['macro_names'][idx])
ax.set_title(f'{state_key}: 宏观时间序列')
ax.set_xlabel('时间步')
ax.set_ylabel('宏观变量')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 3.8 宏观层拟合

在宏观层上重复一次微观层建模步骤，拟合

$$
\mathbf{z}_{t+\tau} \approx \mathbf{A}_z \mathbf{z}_t + \boldsymbol{\varepsilon}_t^{(z)}.
$$

如果粗粒化是合理的，那么宏观层不仅维度更低，而且应当仍能保留较好的动力学闭合性。


In [ ]:
run = state_runs[state_key]
macro_fit = fit_linear_gis_from_pairs(run['Z_now'], run['Z_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_macro = macro_fit['A']
Sigma_macro = macro_fit['Sigma']
run.update({'macro_fit': macro_fit, 'A_macro': A_macro, 'Sigma_macro': Sigma_macro})

plot_matrix_heatmap(A_macro, f'{state_key}: 宏观层 A', row_labels=run['macro_names'], col_labels=run['macro_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_macro, f'{state_key}: 宏观层 Sigma', row_labels=run['macro_names'], col_labels=run['macro_names'], center=None, max_ticks=config['max_ticks'])


### 3.9 宏观层预测、误差

宏观层也做与微观层完全对应的单步与多步预测。这里需要重点比较两个问题：

1. 宏观误差是否显著大于微观误差。
2. 降维之后是否仍保持较好的滚动预测稳定性。

如果 `CE` 虽然提高，但预测误差明显恶化，那么该宏观表示仍然不能算是一个可信的因果涌现候选。


In [ ]:
run = state_runs[state_key]
macro_errors = compute_prediction_errors(run['A_macro'], run['z_data'], tau=config['tau'], horizons=config['horizons'])
run['macro_errors'] = macro_errors

display(build_error_table(macro_errors))
plot_prediction_comparison(macro_errors[1]['predictions'], macro_errors[1]['targets'], run['macro_names'], f'{state_key}: 宏观层单步预测对比')


### 3.10 宏观层GIS指标

宏观层同样需要计算 `Gamma`、`log Gamma`、`J_\alpha`、`D`、`N`。只有在两层都得到完整指标之后，`CE` 才能被解释为真正的“宏观效率增益”，而不是单纯的降维结果。


In [ ]:
run = state_runs[state_key]
macro_metrics = compute_gis_metrics(run['A_macro'], run['Sigma_macro'], alpha=config['alpha'], eps=config['eps'])
run['macro_metrics'] = macro_metrics

display(build_metric_table(macro_metrics))
plot_dual_singular_spectra(
    macro_metrics['sv_forward'],
    macro_metrics['sv_backward'],
    title=f'{state_key}: 宏观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 3.11 CE值+宏微观曲线对比

因果涌现强度这里直接定义为

$$
\mathrm{CE} = J_{\alpha,z} - J_{\alpha,o}.
$$

若 `CE > 0`，说明宏观层在“单位维度效率”意义上优于微观层；但是否值得接受，还要结合前面预测误差是否显著恶化一起看。

这一块把 `CE` 数值和宏微观代表性曲线放到一起展示，便于同时观察“效率提升”和“动态形态是否保留”。


In [ ]:
run = state_runs[state_key]
ce_result = compute_ce_from_micro_macro(run['micro_metrics'], run['macro_metrics'])
run['ce_result'] = ce_result

comparison_df_local = pd.DataFrame([
    {'layer': 'micro', 'J_alpha': run['micro_metrics']['J_alpha'], 'log_Gamma': run['micro_metrics']['log_Gamma'], 'E1': run['micro_errors'][1]['mean_error']},
    {'layer': 'macro', 'J_alpha': run['macro_metrics']['J_alpha'], 'log_Gamma': run['macro_metrics']['log_Gamma'], 'E1': run['macro_errors'][1]['mean_error']},
    {'layer': 'macro - micro', 'J_alpha': ce_result['delta_J'], 'log_Gamma': ce_result['delta_log_Gamma'], 'E1': run['macro_errors'][1]['mean_error'] - run['micro_errors'][1]['mean_error']},
])
print(f'{state_key} 的 CE = {ce_result["CE"]:.6f}')
display(comparison_df_local)

micro_curve_indices = run['obs_indices'][:min(4, len(run['obs_indices']))]
micro_curve_data = run['obs_data'][:, micro_curve_indices]
micro_curve_names = [run['feature_names'][idx] for idx in micro_curve_indices]
plot_macro_micro_curves(micro_curve_data, run['z_data'], micro_curve_names, run['macro_names'], f'{state_key}: 宏微观曲线对比', max_micro=4, max_macro=4, steps=config['curve_window'])


### 3.12 结果汇总

最后把本状态的参数、同步指标、宏微观误差和 `GIS` 指标整理为统一摘要。这样一方面便于本部分回顾，另一方面也能直接服务于最后的四状态横向比较。


In [ ]:
run = state_runs[state_key]
summary_dict, summary_row = summarize_pipeline_results(
    config={'experiment_name': config['experiment_name'], 'state': state_key, 'tau': config['tau'], 'alpha': config['alpha'], 'noise_scale': config['noise_scale'], 'observable': 'identity + square (x-channel)'},
    micro_fit=run['micro_fit'],
    macro_fit=run['macro_fit'],
    micro_metrics=run['micro_metrics'],
    macro_metrics=run['macro_metrics'],
    prediction_results={'micro_errors': run['micro_errors'], 'macro_errors': run['macro_errors']},
    ce_result=run['ce_result'],
    extra={'W': run['W'], 'rank_meta': run['rank_meta'], 'sync_metrics': run['sync_metrics']},
)
state_row = summarize_state_outputs(state_key, run['state_cfg'], run['sync_metrics'], run['micro_metrics'], run['macro_metrics'], run['micro_errors'], run['macro_errors'], run['ce_result'], run['rank_meta'])
run['summary_dict'] = summary_dict
run['summary_row'] = state_row
summary_rows.append(state_row)

display(pd.DataFrame([state_row]))


## 第四部分：D部分


### 4.1 数据生成加上噪音，并绘图

这一块延续 `exp_map_0321.ipynb` 的状态参数，先生成对应的 Rulkov map 原始轨迹，再在提取出的 `x` 通道微观矩阵上加入高斯噪声。

这样做对应研究框架里的“数据准备”步骤：我们先得到统一的原始时间序列 $\{\mathbf{x}_t\}$，再把噪声视作有限观测与未建模扰动的经验代理。这里保留参考 notebook 的同步指标，并额外画出少量代表性通道的“干净/含噪”曲线对比。


In [ ]:
state_key = 'D'
state_cfg = config['states'][state_key]
state_args = dict(state_cfg)
state_args.pop('label', None)

data = generate_two_population_neuron_data(**state_args)
raw_pack = extract_state_matrix_from_rulkov_data(data, include_x=config['include_x'], include_y=config['include_y'], use_transient=config['use_transient'])

x_clean = np.asarray(raw_pack['x_data_raw'], dtype=float)
state_names = list(raw_pack['state_names'])
time_data = np.asarray(raw_pack['time_data'])
sync_metrics = raw_pack['sync_metrics']
noise_seed = config['noise_seed'] + state_cfg['seed']
x_noisy = add_gaussian_noise(x_clean, noise_scale=config['noise_scale'], random_state=noise_seed)['noisy_data']
raw_indices = pick_valid_indices(config['selected_raw_indices'], x_clean.shape[1])

state_runs[state_key] = {'state_cfg': state_cfg, 'data': data, 'raw_pack': raw_pack, 'x_clean': x_clean, 'x_noisy': x_noisy, 'state_names': state_names, 'time_data': time_data, 'sync_metrics': sync_metrics, 'raw_indices': raw_indices}

print('状态:', state_key, '-', state_cfg['label'])
print('原始矩阵形状:', x_clean.shape)
print('噪声种子:', noise_seed)
print('同步指标:', sync_metrics)
display(pd.DataFrame([{'state': state_key, 'sync_state': sync_metrics.get('sync_state'), 'R_a': sync_metrics.get('R_a'), 'R_b': sync_metrics.get('R_b'), 'R_t': sync_metrics.get('R_t'), 'R_delta': sync_metrics.get('R_delta')}]))

plot_neuron_analysis_combo(data, figsize=(18, 7))

fig, ax = plt.subplots(figsize=(10, 4.2))
window = min(config['curve_window'], x_clean.shape[0])
for idx in raw_indices[:4]:
    ax.plot(x_clean[:window, idx], linewidth=1.1, label=f'clean: {state_names[idx]}')
    ax.plot(x_noisy[:window, idx], '--', linewidth=1.1, alpha=0.9, label=f'noisy: {state_names[idx]}')
ax.set_title(f'{state_key}: 原始通道加噪前后对比')
ax.set_xlabel('时间步')
ax.set_ylabel('取值')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 4.2 观测函数、时间尺度与样本配对

这里采用本实验指定的观测函数

$$
\mathbf{o}_t = g(\mathbf{x}_t)=\bigl[\mathbf{x}_t,\ \mathbf{x}_t^{\odot 2}\bigr],
$$

并在固定时间尺度 $\tau=1$ 下构造样本对 $(\mathbf{o}_t, \mathbf{o}_{t+\tau})$。平方项的作用是把最基本的非线性信息直接显式写入观测层，使后续线性 `GIS` 能在更丰富的特征空间中拟合有效动力学。


In [ ]:
run = state_runs[state_key]
obs_pack = build_identity_square_observables(run['x_noisy'], run['state_names'])
obs_data = obs_pack['data']
feature_names = obs_pack['feature_names']
X_now, X_next = prepare_time_pairs(obs_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
obs_indices = pick_valid_indices(config['selected_obs_indices'], obs_data.shape[1], fallback_count=6)

run.update({'obs_pack': obs_pack, 'obs_data': obs_data, 'feature_names': feature_names, 'X_now': X_now, 'X_next': X_next, 'obs_indices': obs_indices})

print('观测层维度:', obs_pack['micro_dim'])
print('样本对形状:', X_now.shape, X_next.shape)
print('前 10 个观测标签:', feature_names[:10])
display(pd.DataFrame([{'state': state_key, 'base_dim': obs_pack['base_dim'], 'micro_dim': obs_pack['micro_dim'], 'tau': config['tau'], 'pair_count': X_now.shape[0]}]))


### 4.3 微观层 A/K_raw 与 Sigma 的拟合

根据数据驱动 `GIS` 的主流程，在观测层拟合

$$
\mathbf{o}_{t+\tau} \approx \mathbf{A}_o \mathbf{o}_t + \boldsymbol{\varepsilon}_t^{(o)},
\qquad
\boldsymbol{\Sigma}_o = \mathrm{Cov}(\mathbf{e}_t^{(o)}).
$$

这里同时保留 `A` 与 `K_raw`。在当前最小二乘实现下，两者本质上是同一经验回归关系的两种记法；热力图主要帮助观察微观层中哪些通道之间存在更强的耦合结构。


In [ ]:
run = state_runs[state_key]
micro_fit = fit_linear_gis_from_pairs(run['X_now'], run['X_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_micro = micro_fit['A']
K_raw_micro = micro_fit['K_raw']
Sigma_micro = micro_fit['Sigma']
run.update({'micro_fit': micro_fit, 'A_micro': A_micro, 'K_raw_micro': K_raw_micro, 'Sigma_micro': Sigma_micro})

print('A_micro shape:', A_micro.shape)
print('K_raw_micro shape:', K_raw_micro.shape)
print('Sigma_micro shape:', Sigma_micro.shape)
plot_matrix_heatmap(A_micro, f'{state_key}: 微观层 A', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(K_raw_micro, f'{state_key}: 微观层 K_raw', row_labels=run['feature_names'], col_labels=run['feature_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_micro, f'{state_key}: 微观层 Sigma', row_labels=run['feature_names'], col_labels=run['feature_names'], center=None, max_ticks=config['max_ticks'])


### 4.4 微观层预测与误差

得到微观层矩阵 $\mathbf{A}_o$ 后，可以做单步和多步滚动预测：

$$
\hat{\mathbf{o}}_{t+k\tau \mid t} = \mathbf{A}_o^k \mathbf{o}_t.
$$

这里重点看两件事：第一，观测层模型是否具有足够好的闭合性；第二，不同状态在相同观测函数下的误差规模是否存在明显差异。


In [ ]:
run = state_runs[state_key]
micro_errors = compute_prediction_errors(run['A_micro'], run['obs_data'], tau=config['tau'], horizons=config['horizons'])
run['micro_errors'] = micro_errors

display(build_error_table(micro_errors))
selected_names = [run['feature_names'][idx] for idx in run['obs_indices'][:4]]
selected_preds = micro_errors[1]['predictions'][:, run['obs_indices'][:4]]
selected_targets = micro_errors[1]['targets'][:, run['obs_indices'][:4]]
plot_prediction_comparison(selected_preds, selected_targets, selected_names, f'{state_key}: 微观层单步预测对比')


### 4.5 微观层 GIS 指标

微观层 `GIS` 指标由 $(\mathbf{A}_o, \boldsymbol{\Sigma}_o)$ 决定，其中核心量包括

$$
D = \log \operatorname{pdet}(\boldsymbol{\Sigma}_o^{-1}),
\qquad
N = \log \operatorname{pdet}(\mathbf{A}_o^\top \boldsymbol{\Sigma}_o^{-1} \mathbf{A}_o),
$$

以及

$$
J_\alpha = \frac{1}{m} \log \Gamma_\alpha^{\mathrm{GIS}}.
$$

其中 $D$ 描述前向确定性，$N$ 描述后向非简并性，$J_\alpha$ 则是维度归一化后的综合效率。


In [ ]:
run = state_runs[state_key]
micro_metrics = compute_gis_metrics(run['A_micro'], run['Sigma_micro'], alpha=config['alpha'], eps=config['eps'])
run['micro_metrics'] = micro_metrics

display(build_metric_table(micro_metrics))
plot_dual_singular_spectra(
    micro_metrics['sv_forward'],
    micro_metrics['sv_backward'],
    title=f'{state_key}: 微观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 4.6 宏观维度选择与 W 的构造

宏观层的关键在于先选维数 $r$，再构造粗粒化矩阵 $\mathbf{W}$。这里沿用 `SVD` 路线，根据微观层后向谱自动选择候选维数，并构造

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t,
\qquad
\mathbf{W} \in \mathbb{R}^{r \times m}.
$$

`W` 的每一行都可以理解为一个宏观变量在微观观测基上的线性组合。由于微观维数较高，这里的标签会自动稀疏显示。


In [ ]:
run = state_runs[state_key]
manual_r = config['manual_ranks'][state_key]
r, rank_meta = choose_macro_rank(run['micro_metrics']['sv_backward'], manual_r=manual_r, eps=config['eps'])
w_result = build_w_from_svd(run['A_micro'], run['Sigma_micro'], r=r, alpha=config['alpha'], eps=config['eps'], mode='two_stage')
W = w_result['W']
macro_names = [f'z_{i+1}' for i in range(r)]
run.update({'rank_meta': rank_meta, 'w_result': w_result, 'W': W, 'macro_names': macro_names})

print('宏观维度选择结果:', rank_meta)
plot_matrix_heatmap(np.abs(W), f'{state_key}: |W| 热力图', row_labels=macro_names, col_labels=run['feature_names'], center=None, figsize=(7.5, 3.8), cmap='Blues', max_ticks=config['max_ticks'])


### 4.7 宏观数据生成

有了粗粒化矩阵之后，就可以把微观观测投影到宏观层，得到

$$
\mathbf{z}_t = \mathbf{W}\mathbf{o}_t.
$$

这一块只负责生成宏观数据和宏观层样本对，并画出少量宏观变量的时间曲线，帮助直观理解粗粒化后的主导模态。


In [ ]:
run = state_runs[state_key]
z_data = apply_coarse_graining(run['W'], run['obs_data'])
Z_now, Z_next = prepare_time_pairs(z_data, tau=config['tau'], burn_in=config['burn_in'], stride=config['stride'])
run.update({'z_data': z_data, 'Z_now': Z_now, 'Z_next': Z_next})

print('宏观数据形状:', z_data.shape)
print('宏观样本对形状:', Z_now.shape, Z_next.shape)

fig, ax = plt.subplots(figsize=(10, 4.0))
for idx in range(min(z_data.shape[1], 4)):
    ax.plot(z_data[:config['curve_window'], idx], linewidth=1.6, label=run['macro_names'][idx])
ax.set_title(f'{state_key}: 宏观时间序列')
ax.set_xlabel('时间步')
ax.set_ylabel('宏观变量')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


### 4.8 宏观层拟合

在宏观层上重复一次微观层建模步骤，拟合

$$
\mathbf{z}_{t+\tau} \approx \mathbf{A}_z \mathbf{z}_t + \boldsymbol{\varepsilon}_t^{(z)}.
$$

如果粗粒化是合理的，那么宏观层不仅维度更低，而且应当仍能保留较好的动力学闭合性。


In [ ]:
run = state_runs[state_key]
macro_fit = fit_linear_gis_from_pairs(run['Z_now'], run['Z_next'], fit_intercept=False, ridge=config['ridge'], regularization=config['eps'])
A_macro = macro_fit['A']
Sigma_macro = macro_fit['Sigma']
run.update({'macro_fit': macro_fit, 'A_macro': A_macro, 'Sigma_macro': Sigma_macro})

plot_matrix_heatmap(A_macro, f'{state_key}: 宏观层 A', row_labels=run['macro_names'], col_labels=run['macro_names'], max_ticks=config['max_ticks'])
plot_matrix_heatmap(Sigma_macro, f'{state_key}: 宏观层 Sigma', row_labels=run['macro_names'], col_labels=run['macro_names'], center=None, max_ticks=config['max_ticks'])


### 4.9 宏观层预测、误差

宏观层也做与微观层完全对应的单步与多步预测。这里需要重点比较两个问题：

1. 宏观误差是否显著大于微观误差。
2. 降维之后是否仍保持较好的滚动预测稳定性。

如果 `CE` 虽然提高，但预测误差明显恶化，那么该宏观表示仍然不能算是一个可信的因果涌现候选。


In [ ]:
run = state_runs[state_key]
macro_errors = compute_prediction_errors(run['A_macro'], run['z_data'], tau=config['tau'], horizons=config['horizons'])
run['macro_errors'] = macro_errors

display(build_error_table(macro_errors))
plot_prediction_comparison(macro_errors[1]['predictions'], macro_errors[1]['targets'], run['macro_names'], f'{state_key}: 宏观层单步预测对比')


### 4.10 宏观层GIS指标

宏观层同样需要计算 `Gamma`、`log Gamma`、`J_\alpha`、`D`、`N`。只有在两层都得到完整指标之后，`CE` 才能被解释为真正的“宏观效率增益”，而不是单纯的降维结果。


In [ ]:
run = state_runs[state_key]
macro_metrics = compute_gis_metrics(run['A_macro'], run['Sigma_macro'], alpha=config['alpha'], eps=config['eps'])
run['macro_metrics'] = macro_metrics

display(build_metric_table(macro_metrics))
plot_dual_singular_spectra(
    macro_metrics['sv_forward'],
    macro_metrics['sv_backward'],
    title=f'{state_key}: 宏观层前向/后向奇异值谱',
    forward_label='前向谱（Sigma^{-1}）',
    backward_label='后向谱（A^T Sigma^{-1} A）',
)


### 4.11 CE值+宏微观曲线对比

因果涌现强度这里直接定义为

$$
\mathrm{CE} = J_{\alpha,z} - J_{\alpha,o}.
$$

若 `CE > 0`，说明宏观层在“单位维度效率”意义上优于微观层；但是否值得接受，还要结合前面预测误差是否显著恶化一起看。

这一块把 `CE` 数值和宏微观代表性曲线放到一起展示，便于同时观察“效率提升”和“动态形态是否保留”。


In [ ]:
run = state_runs[state_key]
ce_result = compute_ce_from_micro_macro(run['micro_metrics'], run['macro_metrics'])
run['ce_result'] = ce_result

comparison_df_local = pd.DataFrame([
    {'layer': 'micro', 'J_alpha': run['micro_metrics']['J_alpha'], 'log_Gamma': run['micro_metrics']['log_Gamma'], 'E1': run['micro_errors'][1]['mean_error']},
    {'layer': 'macro', 'J_alpha': run['macro_metrics']['J_alpha'], 'log_Gamma': run['macro_metrics']['log_Gamma'], 'E1': run['macro_errors'][1]['mean_error']},
    {'layer': 'macro - micro', 'J_alpha': ce_result['delta_J'], 'log_Gamma': ce_result['delta_log_Gamma'], 'E1': run['macro_errors'][1]['mean_error'] - run['micro_errors'][1]['mean_error']},
])
print(f'{state_key} 的 CE = {ce_result["CE"]:.6f}')
display(comparison_df_local)

micro_curve_indices = run['obs_indices'][:min(4, len(run['obs_indices']))]
micro_curve_data = run['obs_data'][:, micro_curve_indices]
micro_curve_names = [run['feature_names'][idx] for idx in micro_curve_indices]
plot_macro_micro_curves(micro_curve_data, run['z_data'], micro_curve_names, run['macro_names'], f'{state_key}: 宏微观曲线对比', max_micro=4, max_macro=4, steps=config['curve_window'])


### 4.12 结果汇总

最后把本状态的参数、同步指标、宏微观误差和 `GIS` 指标整理为统一摘要。这样一方面便于本部分回顾，另一方面也能直接服务于最后的四状态横向比较。


In [ ]:
run = state_runs[state_key]
summary_dict, summary_row = summarize_pipeline_results(
    config={'experiment_name': config['experiment_name'], 'state': state_key, 'tau': config['tau'], 'alpha': config['alpha'], 'noise_scale': config['noise_scale'], 'observable': 'identity + square (x-channel)'},
    micro_fit=run['micro_fit'],
    macro_fit=run['macro_fit'],
    micro_metrics=run['micro_metrics'],
    macro_metrics=run['macro_metrics'],
    prediction_results={'micro_errors': run['micro_errors'], 'macro_errors': run['macro_errors']},
    ce_result=run['ce_result'],
    extra={'W': run['W'], 'rank_meta': run['rank_meta'], 'sync_metrics': run['sync_metrics']},
)
state_row = summarize_state_outputs(state_key, run['state_cfg'], run['sync_metrics'], run['micro_metrics'], run['macro_metrics'], run['micro_errors'], run['macro_errors'], run['ce_result'], run['rank_meta'])
run['summary_dict'] = summary_dict
run['summary_row'] = state_row
summary_rows.append(state_row)

display(pd.DataFrame([state_row]))


## 第五部分：结尾统一摘要

### 5.1 四种情况下的关键指标对比，比如CE，r等等

这一部分把四种状态的关键单值指标放到同一张表里。重点关注：

1. 哪些状态的 `CE` 为正。
2. 哪些状态在 `CE` 改善的同时还能维持较低预测误差。
3. 宏观维数 `r` 是否与同步结构的复杂程度相匹配。


In [ ]:
comparison_df = pd.DataFrame(summary_rows)
comparison_df = comparison_df[['state', 'label', 'sync_state', 'selected_r', 'micro_dim', 'macro_dim', 'micro_J_alpha', 'macro_J_alpha', 'micro_E1', 'macro_E1', 'CE', 'R_t', 'R_delta']]
comparison_df = comparison_df.rename(columns={'state': '状态', 'label': '标签', 'sync_state': '同步判定', 'selected_r': '宏观维度 r', 'micro_dim': '微观维度', 'macro_dim': '宏观维度', 'micro_J_alpha': '微观 J_alpha', 'macro_J_alpha': '宏观 J_alpha', 'micro_E1': '微观 E1', 'macro_E1': '宏观 E1', 'CE': 'CE', 'R_t': 'R_t', 'R_delta': 'R_delta'})
display(comparison_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))
axes[0].bar(comparison_df['状态'], comparison_df['CE'], color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red'])
axes[0].set_title('四种状态的 CE')
axes[0].set_xlabel('状态')
axes[0].set_ylabel('CE')

axes[1].bar(comparison_df['状态'], comparison_df['宏观维度 r'], color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red'])
axes[1].set_title('自动选择的宏观维度')
axes[1].set_xlabel('状态')
axes[1].set_ylabel('r')

x = np.arange(len(comparison_df))
width = 0.36
axes[2].bar(x - width / 2, comparison_df['微观 E1'], width=width, label='微观 E1')
axes[2].bar(x + width / 2, comparison_df['宏观 E1'], width=width, label='宏观 E1')
axes[2].set_xticks(x)
axes[2].set_xticklabels(comparison_df['状态'])
axes[2].set_title('单步预测误差对比')
axes[2].set_xlabel('状态')
axes[2].set_ylabel('E1')
axes[2].legend()

plt.tight_layout()
plt.show()


### 5.2 统一结论（主要是用解释块对整个实验进行汇总解释，总结结论，简单打印一些数值结果就行）

从研究框架的角度看，这个实验主要回答的是：在相同时间尺度、相同噪声强度和相同观测函数设定下，不同同步状态是否会对应不同的宏观效率提升模式。

若某个状态同时满足下面两点：

1. `CE > 0`，说明宏观层在单位维度上更高效。
2. 宏观层的 `E1` 与多步误差没有明显恶化，说明这种效率提升并不是靠牺牲闭合性换来的。

那么就可以认为该状态下出现了更可信的宏观有效表示。反过来，如果 `CE` 虽然为正但预测误差明显变差，或者 `CE` 本身为负，则说明当前粗粒化还不足以支持“可信的因果涌现”判断。


In [ ]:
best_ce_row = comparison_df.sort_values('CE', ascending=False).iloc[0]
lowest_macro_error_row = comparison_df.sort_values('宏观 E1', ascending=True).iloc[0]

print('统一结论摘要：')
print(f"CE 最高的状态: {best_ce_row['状态']}，CE = {best_ce_row['CE']:.6f}，r = {int(best_ce_row['宏观维度 r'])}")
print(f"宏观单步误差最低的状态: {lowest_macro_error_row['状态']}，宏观 E1 = {lowest_macro_error_row['宏观 E1']:.6f}")

for _, row in comparison_df.iterrows():
    trend = '宏观更优' if row['CE'] > 0 else '宏观未优于微观'
    print(f"{row['状态']}: CE={row['CE']:.6f}, 微观E1={row['微观 E1']:.6f}, 宏观E1={row['宏观 E1']:.6f}, r={int(row['宏观维度 r'])}, 判断={trend}")
